# Linked Nodes

```{contents}
:local:
:depth: 2
```

A linked list stores a sequence as separate node objects connected by references. The representation makes insertion and deletion local, but finding a position requires traversal.


Replacing a caller head reference uses ref; review {ref}`Chapter 5 parameter passing <gap-return-values-out-and-ref>`.


```{index} node
```
```{index} linked list
```



```{index} Nodes and References
```

## Nodes and References



Each node stores a value and a reference to the next node.

```text
head -> [10 | next] -> [20 | next] -> [30 | null]
```

The list variable stores a reference to the first node, often called the **head**. The last node points to `null`.

A node is a small class with two members, and the whole list is built from nothing but nodes:

```csharp
public class Node<T>
{
    public T Value { get; }          // the stored item, fixed at construction
    public Node<T>? Next { get; set; }   // the following node, or null at the end
}
```

The `?` on `Next` says that `null` is an expected value, not an error: the last node's `Next` is always `null`. Chapter 11 introduced that annotation. The example links three nodes by hand, assigning each `Next` after the node is created:


In [ ]:
IntroNode<int> first = new IntroNode<int>(10);
first.Next = new IntroNode<int>(20);
first.Next.Next = new IntroNode<int>(30);

Console.WriteLine(first.Value);
Console.WriteLine(first.Next.Value);

public class IntroNode<T>
{
    public T Value { get; }
    public IntroNode<T>? Next { get; set; }

    public IntroNode(T value)
    {
        Value = value;
    }
}

10
20


The output is `10` and then `20`. `first.Next.Next.Next` is `null`, so `first.Next.Next.Next.Value` would throw `NullReferenceException`; every walk along `Next` references has to stop at `null`, which is what the traversal loop in the next section does. Notice that there is no array anywhere: the three nodes can sit anywhere in memory, and only the references connect them.


```{index} alias; linked list; reference
```



```{index} References Are Paths, Not Copies
```

## References Are Paths, Not Copies



Work orders may also be represented by separate node objects. A node's `Next` is a reference to another node, not an embedded copy of the remaining list. Assigning `current = head` copies a reference: both variables initially identify the same object. Changing `current` to `current.Next` moves a cursor without moving `head`.

```text
head ----> [WO-10 | next] ---> [WO-30 | null]
current --^
```

If you traverse by repeatedly assigning to `head`, you lose your entry point to the sequence. A separate cursor preserves the list. Also distinguish changing a reference from changing an object: setting `current = current.Next` moves the cursor; setting `current.Next = ...` changes a link in the sequence.

**Trace (basic):** Draw both arrows before and after one cursor advance. Which variable still reaches the first work order?

```{index} traversal; linked list
```



```{index} Traversal
```

## Traversal



A linked list cannot jump directly to index `7`. To find an item, code follows references one node at a time. This is called **traversal**.

The traversal loop has the same three parts as a counting `for` loop, with a reference in place of an index:

```csharp
for (Node<T>? current = head; current is not null; current = current.Next)
{
    // use current.Value
}
```

Start at the head, continue while there is a node, and advance by following `Next`. The loop ends when `current` becomes `null`, which happens one step past the last node.


In [ ]:
TraversalNode<string> head = new TraversalNode<string>("alpha");
head.Next = new TraversalNode<string>("beta");
head.Next.Next = new TraversalNode<string>("gamma");

int visits = 0;
for (TraversalNode<string>? current = head; current is not null; current = current.Next)
{
    visits++;
    Console.WriteLine($"visit {visits}: {current.Value}");
}

public class TraversalNode<T>
{
    public T Value { get; }
    public TraversalNode<T>? Next { get; set; }
    public TraversalNode(T value) => Value = value;
}

visit 1: alpha
visit 2: beta
visit 3: gamma


The output is three lines, `visit 1: alpha` through `visit 3: gamma`. Reaching `gamma` cost three visits because there is no way to jump to it; an array would have read index 2 directly. Two mistakes produce wrong loops: testing `current.Next is not null` instead of `current is not null` skips the last node, and forgetting the `current = current.Next` step never ends.


```{index} Guided Practice: Count the Nodes
```

### Guided Practice: Count the Nodes


Complete `Count` with the traversal loop so the output is `3`, and make sure an empty list (`null`) returns `0` without throwing.


In [ ]:
using System;

CountNode<string> head = new CountNode<string>("alpha");
head.Next = new CountNode<string>("beta");
head.Next.Next = new CountNode<string>("gamma");

Console.WriteLine(Count(head));
Console.WriteLine(Count<string>(null));   // no argument to infer T from, so name it

static int Count<T>(CountNode<T>? head)
{
    // Your code starts here.
    return 0;
    // Your code ends here.
}

public class CountNode<T>
{
    public T Value { get; }
    public CountNode<T>? Next { get; set; }
    public CountNode(T value) => Value = value;
}


In [ ]:
using System;

// Solution
CountNode<string> head = new CountNode<string>("alpha");
head.Next = new CountNode<string>("beta");
head.Next.Next = new CountNode<string>("gamma");

Console.WriteLine(Count(head));
Console.WriteLine(Count<string>(null));   // no argument to infer T from, so name it

static int Count<T>(CountNode<T>? head)
{
    int count = 0;
    for (CountNode<T>? current = head; current is not null; current = current.Next)
    {
        count++;
    }
    return count;
}

public class CountNode<T>
{
    public T Value { get; }
    public CountNode<T>? Next { get; set; }
    public CountNode(T value) => Value = value;
}


3
0


The loop handles the empty list for free: `current` starts as `null`, the condition fails at once, and `count` stays 0. A list keeps no stored count unless its class maintains one, which is why Chapter 14's complete sequence class tracks `Count` on every insert and delete.


```{index} linked list; insertion invariant; cycle
```



```{index} Insertion Order and Lost Links
```

## Insertion Order and Lost Links



You know the node for `WO-10` and want to insert `WO-20` before `WO-30`. Preserve the old successor before overwriting its only incoming list link.

| Step | References | What remains reachable |
| --- | --- | --- |
| Create B | `A.Next = C`, `B.Next = null` | A still reaches C. |
| Save successor | `B.Next = A.Next` | B now reaches C too. |
| Connect B | `A.Next = B` | A reaches B, then C. |

The correct two assignments are `inserted.Next = node.Next; node.Next = inserted;`. Reversing them makes `inserted.Next` refer to `inserted` itself. The result is a cycle, and the original successor is disconnected from this chain. A traversal waiting for `null` would never finish.

For an empty list there is no predecessor. Inserting at the front sets `newNode.Next = head` before replacing `head`. The same steps work when the old head is null. A list wrapper is useful because it owns the head reference and can update it safely.

**Debug (applied):** Explain the reversed-assignment bug using named nodes A, B, and C. Do not run an unbounded traversal of the broken chain.

```{index} insertion; linked list
```



```{index} Insertion After a Node
```

## Insertion After a Node



If you already have a reference to a node, inserting after it is local. Create a new node, point it to the old next node, then point the current node to the new node.

```text
before: [A] -> [C]
after:  [A] -> [B] -> [C]
```

The order of the two assignments is the whole technique. Point the new node at the old successor first; only then redirect the current node. Reversing them loses the rest of the list, because the only reference to `C` was `node.Next`, and overwriting it first leaves nothing to copy.


In [ ]:
static void InsertAfter<T>(InsertNode<T> node, T value)
{
    InsertNode<T> inserted = new InsertNode<T>(value);
    inserted.Next = node.Next;
    node.Next = inserted;
}

InsertNode<string> head = new InsertNode<string>("A");
head.Next = new InsertNode<string>("C");
InsertAfter(head, "B");

for (InsertNode<string>? current = head; current is not null; current = current.Next)
{
    Console.WriteLine(current.Value);
}

public class InsertNode<T>
{
    public T Value { get; }
    public InsertNode<T>? Next { get; set; }
    public InsertNode(T value) => Value = value;
}

A
B
C


The output is `A`, `B`, `C`. No node moved: `A` still sits where it was, and `C` is reached through `B` now. Trace the reversed order to see the defect: `node.Next = inserted` first makes `A` point at `B`, and then `inserted.Next = node.Next` sets `B.Next` to `B` itself, a loop that drops `C` and never ends when traversed.


```{index} deletion; linked list
```



```{index} Deletion After a Node
```

## Deletion After a Node



Deleting after a known node is also local. Skip over the next node by changing one reference.

```text
before: [A] -> [B] -> [C]
after:  [A] --------> [C]
```

The method returns `false` when there is nothing after the given node, so the caller can tell "nothing removed" from "removed". Removing the *first* node is different, because no node precedes the head; the Removal, Head, and Tail Invariants section below handles that case.


In [ ]:
static bool DeleteAfter<T>(DeleteNode<T> node)
{
    if (node.Next is null) return false;
    node.Next = node.Next.Next;
    return true;
}

DeleteNode<string> head = new DeleteNode<string>("A");
head.Next = new DeleteNode<string>("B");
head.Next.Next = new DeleteNode<string>("C");

DeleteAfter(head);

for (DeleteNode<string>? current = head; current is not null; current = current.Next)
{
    Console.WriteLine(current.Value);
}

public class DeleteNode<T>
{
    public T Value { get; }
    public DeleteNode<T>? Next { get; set; }
    public DeleteNode(T value) => Value = value;
}

A
C


The output is `A`, `C`. Node `B` still exists for a moment, still pointing at `C`, but nothing points at `B` any more, so it is unreachable and the runtime reclaims it. The only reference that changed is `A.Next`. Contrast this with an array, where removing the middle element means shifting every later element one slot to the left.


```{index} linked list; removal; head; tail
```



```{index} Removal, Head, and Tail Invariants
```

## Removal, Head, and Tail Invariants



To remove B after A, preserve B's successor by setting `A.Next = B.Next`. Removing the first node instead requires `head = head.Next`, because there is no preceding node to update. Unlinking does not destroy the object immediately: other references might still reach it, and .NET manages reclamation of unreachable objects.

A wrapper with a tail reference must maintain these rules:

- An empty list has count zero and both head and tail null.
- A nonempty list has a reachable head and tail, and `tail.Next` is null.
- Following `Next` from head visits exactly `Count` nodes and reaches tail last.

Removing the only node must clear both endpoints. Removing the last node of a longer chain must move tail to the predecessor. A tail reference makes append constant-time but does not make removal of the last node constant-time: a singly linked list still has to find that predecessor.

**Predict (applied):** A one-node list is emptied and then appended to. What goes wrong if removal leaves tail pointing to the old node?

```{index} linked list; generic implementation; tail reference
```



```{index} A Complete Singly Linked Sequence
```

## A Complete Singly Linked Sequence



This wrapper presents the same `Add`, `Insert`, indexer, and `RemoveAt` operations as the dynamic array. Nodes are private so client code cannot accidentally create cycles or change count without updating the wrapper.

`NodeAt` validates the index, then follows links. `head!` and `Next!` express facts established by the list invariants after validation; they do not create runtime null checks. If those invariants are broken, the suppression cannot repair them.

Run the cell independently. It inserts the middle order, removes the head, empties the list, and appends again. The last line should be `count=1, first=WO-40`.

In [ ]:
using System;
var orders = new WorkOrderChain<string>();
orders.Add("WO-10");
orders.Add("WO-30");
orders.Insert(1, "WO-20");
for (int i = 0; i < orders.Count; i++) Console.WriteLine(orders[i]);
Console.WriteLine($"removed={orders.RemoveAt(0)}");
orders.RemoveAt(1);
orders.RemoveAt(0);
orders.Add("WO-40");
Console.WriteLine($"count={orders.Count}, first={orders[0]}");
public class WorkOrderChain<T>
{
    private sealed class Node
    {
        public T Value;
        public Node? Next;
        public Node(T value, Node? next = null) { Value = value; Next = next; }
    }
    private Node? head;
    private Node? tail;
    public int Count { get; private set; }
    public T this[int index] => NodeAt(index).Value;
    private Node NodeAt(int index)
    {
        if (index < 0 || index >= Count)
            throw new ArgumentOutOfRangeException(nameof(index));
        Node current = head!;
        for (int i = 0; i < index; i++) current = current.Next!;
        return current;
    }
    public void Add(T value)
    {
        var added = new Node(value);
        if (tail is null) head = added;
        else tail.Next = added;
        tail = added;
        Count++;
    }
    public void Insert(int index, T value)
    {
        if (index < 0 || index > Count)
            throw new ArgumentOutOfRangeException(nameof(index));
        if (index == Count) { Add(value); return; }
        if (index == 0) head = new Node(value, head);
        else
        {
            Node previous = NodeAt(index - 1);
            previous.Next = new Node(value, previous.Next);
        }
        Count++;
    }
    public T RemoveAt(int index)
    {
        if (index < 0 || index >= Count)
            throw new ArgumentOutOfRangeException(nameof(index));
        Node removed;
        if (index == 0)
        {
            removed = head!;
            head = removed.Next;
            if (head is null) tail = null;
        }
        else
        {
            Node previous = NodeAt(index - 1);
            removed = previous.Next!;
            previous.Next = removed.Next;
            if (ReferenceEquals(removed, tail)) tail = previous;
        }
        Count--;
        return removed.Value;
    }
}

WO-10
WO-20
WO-30
removed=WO-10
count=1, first=WO-40


Notice that the interface hides implementation choices. `Insert(1, value)` first locates the predecessor and then rewires links. Although the rewiring is O(1), this index-based operation is O(n) in the worst case. The earlier `InsertAfter` example had a different assumption: its caller already held a node reference.

The demonstration prints by repeated indexing only because the list is tiny. For a long linked list, that pattern repeats traversal: indexes zero through `n - 1` require roughly `0 + 1 + ... + (n - 1)` link advances. A cursor or iterator should visit the sequence once instead.

**Extend (challenge):** Add an iterator using chapter 14's `yield return`. Compare a single traversal with repeated indexer reads. Explain what should happen if clients modify the list during iteration; production collections define such behavior explicitly.

```{index} A Header Node Removes Special Cases
```

### A Header Node Removes Special Cases

Every operation above had a special case for the first node: inserting at the front replaces `head`, removing the first node replaces `head`, and an empty list has no node to point at. Each special case is another branch to write and test. A common simplification is a **header node**, also called a sentinel: a node that is allocated when the list is created, never stores a real value, and always sits in front of the first element. With a header, every real node has a predecessor, so insertion and removal become one case each.

```text
empty:     header -> null
one item:  header -> [WO-10] -> null
```

Insertion before the first element is now "insert after the header," and removal of the first element is "remove after the header." The cost is one extra node of memory per list and the discipline of never counting or printing the header. The example below is a list that offers only two operations, insert after a node and remove after a node, and uses them for every update, including the ones at the front.

In [ ]:
using System;

var orders = new HeaderChain<string>();
orders.InsertAfter(orders.Header, "WO-30");      // front insertion is "after the header"
orders.InsertAfter(orders.Header, "WO-10");
orders.InsertAfter(orders.Header.Next!, "WO-20");
orders.Print("after three inserts");

orders.RemoveAfter(orders.Header);               // front removal is also one case
orders.Print("after removing the first");
orders.RemoveAfter(orders.Header);
orders.RemoveAfter(orders.Header);
orders.Print("after emptying");
Console.WriteLine($"RemoveAfter on empty returns {orders.RemoveAfter(orders.Header)}");

public class HeaderChain<T>
{
    public sealed class Node
    {
        public T Value = default!;
        public Node? Next;
    }
    public Node Header { get; } = new Node();   // allocated once; never holds a real value
    public int Count { get; private set; }

    public void InsertAfter(Node node, T value)
    {
        node.Next = new Node { Value = value, Next = node.Next };
        Count++;
    }

    public bool RemoveAfter(Node node)
    {
        if (node.Next is null) return false;    // the only check: is there a successor?
        node.Next = node.Next.Next;
        Count--;
        return true;
    }

    public void Print(string label)
    {
        var values = new System.Collections.Generic.List<string>();
        for (Node? current = Header.Next; current is not null; current = current.Next)
            values.Add($"{current.Value}");
        Console.WriteLine($"{label}: [{string.Join(", ", values)}] count={Count}");
    }
}

after three inserts: [WO-10, WO-20, WO-30] count=3
after removing the first: [WO-20, WO-30] count=2
after emptying: [] count=0
RemoveAfter on empty returns False


The first line shows `[WO-10, WO-20, WO-30]`: two inserts after the header each pushed a value to the front, and the third insert went after the first real node. The removals then use the same `RemoveAfter(Header)` call three times, with no branch asking whether the list is about to become empty, and the final call on an empty list returns `false` because the header's `Next` is `null`. Compare this with `WorkOrderChain<T>` above, whose `RemoveAt(0)` and `Insert(0, value)` each need an `if` for the head. The header buys that simplicity for one node of memory. The traversal in `Print` starts at `Header.Next`, not at `Header`; starting at the header itself is the one new mistake this design introduces, and it shows up as a phantom default value at the front of every printout.

```{index} Costs and Tradeoffs
```



```{index} Costs and Tradeoffs
```

## Costs and Tradeoffs



Linked lists make some updates cheap only after the position is known.

| Operation | Singly linked list cost |
| --- | --- |
| Read first item | O(1) |
| Traverse all items | O(n) |
| Find by value | O(n) |
| Insert after known node | O(1) |
| Delete after known node | O(1) |
| Read by index | O(n) |

The phrase "known node" matters. If the program must search for the node first, that search is part of the total cost.

Compare the two columns of the costs table with the array's: an array reads by index in O(1) and inserts or deletes in the middle in O(n) because of shifting; the chain reverses both, reading by index in O(n) and updating in O(1) once the node is in hand. Neither side wins the search row: an unsorted sequence must be scanned in either representation. The honest way to read any linked-list claim of "constant-time insertion" is to ask how the program obtained the node reference. If it searched by value or walked to an index, that traversal belongs in the cost. If it kept the reference from an earlier operation, for example the node of the work order it just processed, the constant-time claim stands. Every linked structure later in the book, from queues to graphs, relies on exactly that habit of keeping references rather than recomputing positions.

```{index} doubly linked list; predecessor; memory overhead
```



```{index} Doubly Linked Lists: A Different Tradeoff
```

## Doubly Linked Lists: A Different Tradeoff



A doubly linked node stores both `Next` and `Previous`. A known node can therefore find its predecessor without a search, and a stored tail can support constant-time removal at the end. The price is another reference per node and more links to maintain.

```text
null <- [A] <-> [B] <-> [C] -> null
```

Removing B connects `A.Next` to C and `C.Previous` to A. Removing an endpoint also changes the wrapper's head or tail. This representation still does not provide direct indexing: to find a numeric position, follow links from an endpoint.

**Compare (applied):** For frequent removal of a known item, explain what information singly and doubly linked lists require. Include the cost of locating the item if it is supplied only by value.

Written out, removing a known node `b` from a doubly linked chain is two assignments plus endpoint care:

```csharp
if (b.Previous is not null) b.Previous.Next = b.Next; else head = b.Next;
if (b.Next is not null) b.Next.Previous = b.Previous; else tail = b.Previous;
```

Each line repairs one direction of the chain, and each has an `else` for the case where `b` was an endpoint. The order of the two lines does not matter here because neither overwrites a reference the other still needs; insertion is the delicate one, where the new node's two links must be set before either neighbor is redirected. A header and trailer pair, one sentinel at each end, removes all four `else` branches in the same way the single header did above.

```{index} Code Tracing: Which Values Remain Reachable?
```

### Code Tracing: Which Values Remain Reachable?

Trace the reference assignments below by drawing the nodes after each statement. The nodes start as the chain `A -> B -> C -> D`. Predict the output of the final loop, which walks from `head`, and say which node objects are no longer reachable from `head`.

```csharp
var d = new Node("D"); var c = new Node("C") { Next = d };
var b = new Node("B") { Next = c }; var a = new Node("A") { Next = b };
Node head = a;

a.Next = c;                 // 1
c.Next = new Node("E") { Next = d };   // 2
head = head.Next!;          // 3
for (Node? n = head; n is not null; n = n.Next) Console.Write(n.Value + " ");
```

In [ ]:
// Solution
using System;

var d = new Node("D"); var c = new Node("C") { Next = d };
var b = new Node("B") { Next = c }; var a = new Node("A") { Next = b };
Node head = a;

a.Next = c;                            // 1: A -> C -> D; B is unreachable from head
c.Next = new Node("E") { Next = d };   // 2: A -> C -> E -> D
head = head.Next!;                     // 3: head now starts at C; A is unreachable from head
for (Node? n = head; n is not null; n = n.Next) Console.Write(n.Value + " ");
Console.WriteLine();

public class Node
{
    public string Value;
    public Node? Next;
    public Node(string value) => Value = value;
}

C E D 


The loop prints `C E D`. Statement 1 bypassed `B`, statement 2 spliced `E` between `C` and `D` by giving the new node its successor before redirecting `c.Next`, and statement 3 moved the entry point forward, which abandons `A`. From `head`, the nodes `A` and `B` are unreachable; the local variables `a` and `b` still refer to them in this program, which is why they are not reclaimed yet, but a list wrapper that held only `head` would have lost both. If you predicted `A C E D`, you missed that statement 3 assigns to `head` rather than to a cursor, the mistake that References Are Paths, Not Copies warned about.

```{index} Code Completion: Insert at the Front
```

### Code Completion: Insert at the Front

Complete `PushFront` so that it adds a value at the front of the chain and returns the new head. It must work when the chain is empty, which is why the method returns the head instead of relying on a wrapper. The expected output is `WO-30 WO-20 WO-10` and then `count=3`.

In [ ]:
using System;

FrontNode<string>? head = null;
head = PushFront(head, "WO-10");
head = PushFront(head, "WO-20");
head = PushFront(head, "WO-30");

int count = 0;
for (FrontNode<string>? n = head; n is not null; n = n.Next) { Console.Write(n.Value + " "); count++; }
Console.WriteLine();
Console.WriteLine($"count={count}");

static FrontNode<T> PushFront<T>(FrontNode<T>? head, T value)
{
    // Your code starts here.
    throw new NotImplementedException();
    // Your code ends here.
}

public class FrontNode<T>
{
    public T Value { get; }
    public FrontNode<T>? Next { get; set; }
    public FrontNode(T value) => Value = value;
}

In [ ]:
// Solution
using System;

FrontNode<string>? head = null;
head = PushFront(head, "WO-10");
head = PushFront(head, "WO-20");
head = PushFront(head, "WO-30");

int count = 0;
for (FrontNode<string>? n = head; n is not null; n = n.Next) { Console.Write(n.Value + " "); count++; }
Console.WriteLine();
Console.WriteLine($"count={count}");

static FrontNode<T> PushFront<T>(FrontNode<T>? head, T value)
{
    var added = new FrontNode<T>(value);
    added.Next = head;      // the old first node, or null for an empty chain
    return added;           // the caller stores this as the new head
}

public class FrontNode<T>
{
    public T Value { get; }
    public FrontNode<T>? Next { get; set; }
    public FrontNode(T value) => Value = value;
}

WO-30 WO-20 WO-10 
count=3


The same two lines serve the empty chain and the nonempty one, because `added.Next = head` simply stores `null` when there is no first node yet. The caller must write `head = PushFront(head, ...)`: a version that assigns to its `head` parameter and returns nothing changes only the method's own copy of the reference, and the caller's chain never grows. That is the parameter-passing rule from Chapter 5 applied to references, and the reason a list wrapper that owns `head` is the more robust design.

```{index} Check Your Understanding
```

### Check Your Understanding

Answer each question before opening its dropdown.

1. A singly linked list keeps a `tail` reference. Which of these is still O(n): append, remove the last node, or read the last value?

```{dropdown} Answer
Removing the last node. The tail reference reaches the last node in one step for appending or reading, but removal must set the predecessor's `Next` to `null`, and a singly linked node does not know its predecessor; finding it is a traversal from the head.
```

2. What goes wrong if `RemoveAfter` is written as `node.Next = node.Next.Next;` with no `null` check, and the method is called on the last node?

```{dropdown} Answer
`node.Next` is `null`, so `node.Next.Next` dereferences a null reference and throws `NullReferenceException`. The check `if (node.Next is null) return false;` is the contract's statement that there was nothing to remove.
```

3. Why can a header node make the code shorter without changing any cost in the table?

```{dropdown} Answer
The header removes branches, not work. Insertion and removal still rewire a constant number of links, and reaching a position still costs a traversal. The header guarantees that every real node has a predecessor, so the front case needs no separate code.
```

4. A doubly linked node is 50 percent larger than a singly linked node with the same element. When is that price clearly worth paying?

```{dropdown} Answer
When the program frequently removes or inserts next to a node it already holds, especially at the end of the list, because the `Previous` link turns an O(n) predecessor search into one step. If the workload only appends and walks forward, the extra link buys nothing.
```

```{index} practice
```



```{index} Practice
```

## Practice



1. Draw the references after inserting `B` between `A` and `C`.
2. Explain why linked-list indexed access is O(n).
3. Write a method that counts nodes in a linked list starting from `head`.


```{index} summary
```



```{index} Summary
```

## Summary



A linked list stores sequence order through references rather than contiguous slots. This makes local insertion and deletion simple, but traversal is unavoidable for search and indexed access.


```{rubric} Footnotes
```


[^1402_linked_nodes-1]: Later chapters use linked nodes again for stacks, queues, trees, and graphs.
